# METAGENE-1: Full Fine-Tuning Proof of Concept

## Goal
Get the model to understand Gujarat wastewater sequences as well as it
understands its US training data — targeting CE loss as low as possible,
ideally approaching the training-set reference of 1.24.

## Why This Is Different From Our Previous LoRA Experiments

| | LoRA (v4/v6) | This experiment |
|--|--|--|
| Trainable params | 0.12–0.96% | 100% |
| Frozen layers | FFN, embeddings | Nothing frozen |
| Approach | Adapter insertion | Direct weight update |
| Expected loss floor | ~4.78 | ~1.5–3.0 |
| GPU memory needed | 16GB (T4) | 40GB (A100) |

## Strategy
1. Start from METAGENE-1 base weights
2. Train ALL parameters on Gujarat sequences (language modelling)
3. Use gradient checkpointing + DeepSpeed ZeRO-2 to fit in A100 memory
4. Track training loss — when it approaches 1.24, the model 'understands' our data
5. Save checkpoint — this becomes India-METAGENE v7

## ⚠️ IMPORTANT
- Run on **A100 (40GB)** runtime — T4 will OOM
- This will use ~35GB VRAM
- Each epoch over 200K sequences takes ~4 hours
- Training loss of ~1.24 may require 10–50 epochs over full dataset
- Start with 5 epochs on 500K sequences to prove concept

## Runtime → Change runtime type → A100 GPU before running

In [ ]:
# Cell 1: Install
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'transformers>=4.40', 'accelerate>=0.27', 'safetensors',
    'sentencepiece', 'huggingface_hub', 'bitsandbytes',
    'scipy', 'matplotlib',
], check=True)
import torch
assert torch.cuda.is_available(), 'Need GPU'
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {vram:.0f} GB')
if vram < 35:
    print('WARNING: Need A100 (40GB) for full fine-tuning.')
    print('On T4 (16GB), use gradient_checkpointing + micro_batch=1')
print('Setup complete.')

In [ ]:
# Cell 2: Configuration
HF_TOKEN     = 'PASTE_YOUR_HF_TOKEN_HERE'
HF_USERNAME  = 'YOUR_HF_USERNAME'
DATA_REPO    = f'{HF_USERNAME}/gujarat-wastewater'
OLD_DATA_REPO = 'saurabhthakar3/gujarat-wastewater-shotgun'
MODEL_REPO   = f'{HF_USERNAME}/india-metagene-1'

import os
os.environ['HF_TOKEN'] = HF_TOKEN

# Training config
CFG = {
    # Data
    'n_seqs_per_epoch'  : 200_000,   # sequences per epoch
    'n_epochs'          : 5,          # start small — prove concept
    'max_len'           : 512,        # tokens
    'min_read_len'      : 100,        # bp — Liu's optimal range
    'max_read_len'      : 300,        # bp — Liu's optimal range
    'seed'              : 42,

    # Training
    'micro_batch'       : 1,          # per GPU — A100 can do 2-4
    'grad_accum'        : 32,         # effective batch = 32
    'lr'                : 1e-5,       # conservative — we don't want catastrophic forgetting
    'lr_warmup_steps'   : 100,
    'weight_decay'      : 0.01,
    'grad_clip'         : 1.0,
    'grad_checkpoint'   : True,       # saves ~40% VRAM

    # Saving
    'save_every_steps'  : 500,
    'val_every_steps'   : 500,
    'val_n_reads'       : 2000,
}

print('Configuration:')
for k, v in CFG.items():
    print(f'  {k:<22}: {v}')

In [ ]:
# Cell 3: Load METAGENE-1 — ALL parameters unfrozen
# This is the key difference from LoRA — everything will be updated
import torch, time
from transformers import AutoTokenizer, AutoModelForCausalLM
from pathlib import Path

device = torch.device('cuda')
CACHE  = Path('/content/model_cache')
CACHE.mkdir(exist_ok=True)

print('Loading METAGENE-1 (ALL parameters — no frozen layers)...')
t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(
    'metagene-ai/METAGENE-1', cache_dir=str(CACHE))

# Load in float16 for memory efficiency
# We will cast to float32 only for gradient computation
model = AutoModelForCausalLM.from_pretrained(
    'metagene-ai/METAGENE-1',
    cache_dir=str(CACHE),
    torch_dtype=torch.float16,
    device_map='cuda',
)

# Enable gradient checkpointing — trades compute for VRAM
if CFG['grad_checkpoint']:
    model.gradient_checkpointing_enable()
    print('Gradient checkpointing enabled.')

# ALL parameters trainable — no LoRA, no frozen layers
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Loaded in {time.time()-t0:.0f}s')
print(f'Trainable parameters: {trainable/1e9:.2f}B / {total/1e9:.2f}B (100%)')
print(f'VRAM after load: {torch.cuda.memory_allocated()/1e9:.1f} GB')
print()
print('Compare to LoRA:')
print('  v4 (rank 8)  : 8.4M trainable (0.12%)')
print('  v6 (rank 32) : 67.1M trainable (0.96%)')
print(f'  This         : {trainable/1e9:.1f}B trainable (100%)')

In [ ]:
# Cell 4: Load Gujarat training data
# Filter to 100-300bp reads — Liu's optimal anomaly detection range
import gzip, random
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download, list_repo_files

api = HfApi()

# List all FASTA files in old repo
print('Finding all Gujarat FASTA files...')
try:
    all_files = list(api.list_repo_files(
        repo_id=OLD_DATA_REPO, repo_type='dataset', token=HF_TOKEN))
    fasta_files = [f for f in all_files
                   if f.startswith('phase3_fasta/') and f.endswith('.fasta.gz')]
    print(f'Found {len(fasta_files)} FASTA files')
except Exception as e:
    print(f'Could not list old repo: {e}')
    fasta_files = []

FASTA_DIR = Path('/content/gujarat_fasta')
FASTA_DIR.mkdir(exist_ok=True)

# Download a subset of files for proof of concept
# Use training samples (not validation samples)
VALIDATION_SAMPLES = [
    'AAMO_R1','ADSU_R1','AJWI_R1','AMPW_R1',
    'GJPW_R1','GKMO_R1','GKSU_R1','GVWI_R1',
    'RDSU_R1','RDWI_R1','RPPW_R1','RRMO_R1',
    'VAMO_R1','VASU_R1','VCPW_R1','VCWI_R1',
]

train_files = [
    f for f in fasta_files
    if not any(vs in f for vs in VALIDATION_SAMPLES)
]
val_files = [
    f for f in fasta_files
    if any(vs in f for vs in VALIDATION_SAMPLES)
]

print(f'Training files  : {len(train_files)}')
print(f'Validation files: {len(val_files)}')

def load_seqs_from_hf(hf_path, repo_id, n_max=None,
                       min_len=100, max_len=300):
    """Download and filter sequences."""
    local = FASTA_DIR / Path(hf_path).name
    if not local.exists():
        hf_hub_download(repo_id=repo_id, repo_type='dataset',
                        filename=hf_path, local_dir=str(FASTA_DIR),
                        token=HF_TOKEN)
    seqs = []
    opener = gzip.open if str(local).endswith('.gz') else open
    with opener(local, 'rt') as f:
        seq = []
        for line in f:
            line = line.rstrip()
            if line.startswith('>'):
                if seq:
                    s = ''.join(seq)
                    if min_len <= len(s) <= max_len:
                        seqs.append(s)
                seq = []
            else:
                seq.append(line)
        if seq:
            s = ''.join(seq)
            if min_len <= len(s) <= max_len:
                seqs.append(s)
    if n_max: seqs = seqs[:n_max]
    return seqs

print(f'\nRead length filter: {CFG["min_read_len"]}–{CFG["max_read_len"]} bp')
print('(Matching Liu et al. optimal anomaly detection range)')

In [ ]:
# Cell 5: Full fine-tuning training loop
# Key difference from LoRA: model.parameters() not peft parameters
import torch, time, json, random
import numpy as np
from torch.amp import autocast, GradScaler
from transformers import get_linear_schedule_with_warmup
from huggingface_hub import HfApi

api = HfApi()

# Optimizer — update ALL model parameters
# Use lower lr than LoRA to reduce catastrophic forgetting risk
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG['lr'],
    weight_decay=CFG['weight_decay'],
    betas=(0.9, 0.95),  # standard LLM betas
)

steps_per_epoch = CFG['n_seqs_per_epoch'] // (CFG['micro_batch'] * CFG['grad_accum'])
total_steps     = steps_per_epoch * CFG['n_epochs']

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=CFG['lr_warmup_steps'],
    num_training_steps=total_steps,
)

scaler = GradScaler()  # for mixed precision

print(f'Steps per epoch : {steps_per_epoch}')
print(f'Total steps     : {total_steps}')
print(f'Effective batch : {CFG["micro_batch"] * CFG["grad_accum"]}')
print(f'Learning rate   : {CFG["lr"]}')
print()
print('NOTE: This updates ALL 7B parameters.')
print('Training loss should drop toward 1.24 as model learns Indian WW.')
print()

@torch.no_grad()
def compute_val_loss(val_seqs, n=2000):
    """Compute validation CE loss — same metric as anomaly scoring."""
    model.eval()
    random.shuffle(val_seqs)
    sample  = val_seqs[:n]
    losses  = []
    for i in range(0, len(sample), 8):
        batch  = sample[i:i+8]
        inputs = tokenizer(
            batch, return_tensors='pt', padding=True,
            truncation=True, max_length=CFG['max_len'],
            add_special_tokens=False
        ).to('cuda')
        with autocast('cuda', dtype=torch.float16):
            out = model(**inputs, labels=inputs['input_ids'])
        logits  = out.logits[...,:-1,:].contiguous().float()
        targets = inputs['input_ids'][...,1:].contiguous()
        mask    = inputs['attention_mask'][...,1:].contiguous().float()
        tl = torch.nn.CrossEntropyLoss(reduction='none')(
            logits.view(-1, logits.size(-1)), targets.view(-1)
        ).view(targets.size())
        pr = (tl * mask).sum(1) / mask.sum(1).clamp(min=1)
        losses.extend(pr.cpu().float().tolist())
        del out, inputs; torch.cuda.empty_cache()
    model.train()
    return float(np.mean(losses)), float(np.std(losses))


def save_checkpoint(epoch, step, val_loss, tag='fullft'):
    """Save full model checkpoint to HF."""
    ckpt_dir = Path(f'/content/checkpoint_{tag}_epoch{epoch}_step{step}')
    ckpt_dir.mkdir(exist_ok=True)
    model.save_pretrained(str(ckpt_dir))
    tokenizer.save_pretrained(str(ckpt_dir))
    # Save to HF
    try:
        api.upload_folder(
            folder_path=str(ckpt_dir),
            repo_id=MODEL_REPO, repo_type='model',
            path_in_repo=f'checkpoint_{tag}_epoch{epoch}_step{step}',
            token=HF_TOKEN
        )
        print(f'  ✓ Checkpoint saved to HF')
    except Exception as e:
        print(f'  ✗ HF upload failed: {e}')
        print(f'  Checkpoint saved locally at {ckpt_dir}')


# ── Load validation sequences ──────────────────────────────────
print('Loading validation sequences...')
val_seqs = []
for f in val_files[:4]:  # 4 val files
    try:
        seqs = load_seqs_from_hf(f, OLD_DATA_REPO,
                                  min_len=CFG['min_read_len'],
                                  max_len=CFG['max_read_len'])
        val_seqs.extend(seqs[:500])
    except Exception as e:
        print(f'  Val file failed: {f}: {e}')
print(f'Validation sequences: {len(val_seqs)}')

# Baseline validation loss (before any training)
val_loss_0, val_std_0 = compute_val_loss(val_seqs)
print(f'\nBaseline val loss (before training): {val_loss_0:.4f} ± {val_std_0:.4f}')
print(f'(Compare: LoRA v6 best = 4.7890, target = 1.24)')

BASELINE = val_loss_0
history  = {
    'train_loss': [], 'val_loss': [], 'val_std': [],
    'steps': [], 'epochs': [],
    'baseline': BASELINE, 'lora_v6_best': 4.7890, 'target': 1.24,
    'config': CFG,
}

In [ ]:
# Cell 6: Training — the main loop
import time, random, json
import numpy as np
from pathlib import Path

model.train()
global_step  = 0
best_val     = BASELINE
train_files_shuffled = train_files.copy()

print(f'Starting full fine-tuning')
print(f'Epochs: {CFG["n_epochs"]} | Seqs/epoch: {CFG["n_seqs_per_epoch"]:,}')
print(f'This updates ALL {sum(p.numel() for p in model.parameters())/1e9:.1f}B parameters')
print('─' * 65)

for epoch in range(1, CFG['n_epochs'] + 1):
    print(f'\n=== EPOCH {epoch}/{CFG["n_epochs"]} ===')

    # Sample sequences from training files
    print('Sampling training sequences...')
    random.seed(CFG['seed'] + epoch)
    random.shuffle(train_files_shuffled)

    train_seqs = []
    for fpath in train_files_shuffled:
        if len(train_seqs) >= CFG['n_seqs_per_epoch']:
            break
        try:
            seqs = load_seqs_from_hf(
                fpath, OLD_DATA_REPO,
                min_len=CFG['min_read_len'],
                max_len=CFG['max_read_len']
            )
            train_seqs.extend(seqs)
        except Exception as e:
            continue

    random.shuffle(train_seqs)
    train_seqs = train_seqs[:CFG['n_seqs_per_epoch']]
    print(f'Training sequences this epoch: {len(train_seqs):,}')

    optimizer.zero_grad()
    step_in_epoch  = 0
    epoch_loss_sum = 0.0
    epoch_loss_n   = 0
    t_epoch        = time.time()

    for seq_idx in range(0, len(train_seqs), CFG['micro_batch']):
        batch_seqs = train_seqs[seq_idx:seq_idx + CFG['micro_batch']]

        inputs = tokenizer(
            batch_seqs, return_tensors='pt', padding=True,
            truncation=True, max_length=CFG['max_len'],
            add_special_tokens=False
        ).to('cuda')

        with autocast('cuda', dtype=torch.float16):
            outputs = model(**inputs, labels=inputs['input_ids'])
            loss    = outputs.loss / CFG['grad_accum']

        scaler.scale(loss).backward()
        epoch_loss_sum += outputs.loss.item()
        epoch_loss_n   += 1
        del outputs, inputs
        torch.cuda.empty_cache()

        # Gradient accumulation step
        if (seq_idx // CFG['micro_batch'] + 1) % CFG['grad_accum'] == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), CFG['grad_clip'])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            global_step  += 1
            step_in_epoch += 1

            # Progress report
            if step_in_epoch % 50 == 0:
                avg_loss = epoch_loss_sum / epoch_loss_n
                pct      = seq_idx / len(train_seqs) * 100
                elapsed  = time.time() - t_epoch
                eta_min  = elapsed / max(pct, 1) * (100 - pct) / 60
                print(f'  Step {step_in_epoch:>5} | '
                      f'loss={avg_loss:.4f} | '
                      f'{pct:.1f}% | '
                      f'ETA {eta_min:.0f}min | '
                      f'LR={scheduler.get_last_lr()[0]:.2e}')

            # Validation checkpoint
            if global_step % CFG['val_every_steps'] == 0:
                vl, vs = compute_val_loss(val_seqs, CFG['val_n_reads'])
                improve = BASELINE - vl
                vs_lora = 4.7890 - vl
                print(f'\n  --- Val step {global_step} ---')
                print(f'  Val loss   : {vl:.4f} ± {vs:.4f}')
                print(f'  vs baseline: +{improve:.4f} ({improve/BASELINE*100:.2f}%)')
                print(f'  vs LoRA v6 : {vs_lora:+.4f}')
                print(f'  vs target  : {vl - 1.24:+.4f} (target=1.24)')

                history['val_loss'].append(vl)
                history['val_std'].append(vs)
                history['steps'].append(global_step)

                if vl < best_val:
                    best_val = vl
                    save_checkpoint(epoch, global_step, vl, 'fullft')
                print()
                model.train()

    # End of epoch
    epoch_loss = epoch_loss_sum / epoch_loss_n
    vl, vs     = compute_val_loss(val_seqs, CFG['val_n_reads'])
    improve    = BASELINE - vl
    elapsed_m  = (time.time() - t_epoch) / 60

    print(f'\nEpoch {epoch} done | {elapsed_m:.0f} min')
    print(f'  Train loss : {epoch_loss:.4f}')
    print(f'  Val loss   : {vl:.4f} ± {vs:.4f}')
    print(f'  vs baseline: +{improve:.4f} ({improve/BASELINE*100:.2f}%)')
    print(f'  vs LoRA v6 : {4.7890-vl:+.4f}')
    print(f'  vs target  : {vl - 1.24:+.4f} (target=1.24)')

    history['train_loss'].append(epoch_loss)
    history['epochs'].append(epoch)

    # Save epoch checkpoint
    save_checkpoint(epoch, 'final', vl, 'fullft')

    # Save history
    with open('/content/fullft_history.json', 'w') as f:
        json.dump(history, f, indent=2)

print('\nTraining complete.')
print(f'Best val loss : {best_val:.4f}')
print(f'vs baseline   : +{BASELINE - best_val:.4f}')
print(f'vs LoRA v6    : {4.7890 - best_val:+.4f}')
print(f'vs target     : {best_val - 1.24:+.4f}')

In [ ]:
# Cell 7: Learning curve — shows path from baseline toward 1.24
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Training loss curve
ax = axes[0]
ax.plot(range(1, len(history['train_loss'])+1),
        history['train_loss'], 'b-o', lw=2, ms=8, label='Train loss')
ax.axhline(1.24,     color='green',  ls='--', lw=1.5, label='Target (1.24)')
ax.axhline(BASELINE, color='red',    ls='--', lw=1.5,
           label=f'Baseline ({BASELINE:.3f})')
ax.axhline(4.7890,   color='purple', ls=':',  lw=1.2,
           label='LoRA v6 (4.789)')
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('CE Loss', fontsize=11)
ax.set_title('Full Fine-Tuning: Training Loss',
             fontweight='bold', fontsize=11)
ax.legend(fontsize=9)

# Validation loss curve
ax = axes[1]
if history['steps']:
    ax.errorbar(history['steps'], history['val_loss'],
                yerr=history['val_std'],
                fmt='b-o', lw=2, ms=6, capsize=4, label='Val loss')
ax.axhline(1.24,     color='green',  ls='--', lw=1.5, label='Target (1.24)')
ax.axhline(BASELINE, color='red',    ls='--', lw=1.5,
           label=f'Baseline ({BASELINE:.3f})')
ax.axhline(4.7890,   color='purple', ls=':',  lw=1.2,
           label='LoRA v6 best (4.789)')
ax.set_xlabel('Global Step', fontsize=11)
ax.set_ylabel('Val CE Loss', fontsize=11)
ax.set_title('Full Fine-Tuning: Validation Loss',
             fontweight='bold', fontsize=11)
ax.legend(fontsize=9)

plt.suptitle('India-METAGENE v7: Full Fine-Tuning (100% parameters)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/fullft_curve.pdf', dpi=150, bbox_inches='tight')
plt.savefig('/content/fullft_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Learning curve saved.')

In [ ]:
# Cell 8: Score Gujarat data with fully fine-tuned model
# This is the definitive result — how low can Gujarat CE loss go?
import torch, gzip, random
import numpy as np
from torch.amp import autocast
from huggingface_hub import hf_hub_download
from pathlib import Path

model.eval()

VAL_SAMPLES = [
    ('AAMO_R1', 'Ahmedabad', 'Monsoon'),
    ('ADSU_R1', 'Ahmedabad', 'Summer'),
    ('GJPW_R1', 'Gandhinagar', 'PreWinter'),
    ('GKMO_R1', 'Gandhinagar', 'Monsoon'),
    ('RDWI_R1', 'Rajkot', 'Winter'),
    ('RPPW_R1', 'Rajkot', 'PreWinter'),
    ('VCPW_R1', 'Vadodara', 'PreWinter'),
    ('VCWI_R1', 'Vadodara', 'Winter'),
]

print('Scoring Gujarat validation samples with fully fine-tuned model...')
print('─' * 65)
print(f'{"Sample":<12} {"City":<14} {"Season":<12} {"CE Loss":>12}')
print('─' * 65)

final_results = []
for sample, city, season in VAL_SAMPLES:
    try:
        f = hf_hub_download(
            repo_id=OLD_DATA_REPO, repo_type='dataset',
            filename=f'phase3_fasta/{sample}.fasta.gz',
            local_dir='/content/val_fastas', token=HF_TOKEN
        )
        seqs = []
        with gzip.open(f, 'rt') as fh:
            seq = []
            for line in fh:
                line = line.rstrip()
                if line.startswith('>'):
                    if seq: seqs.append(''.join(seq))
                    seq = []
                else:
                    seq.append(line)
            if seq: seqs.append(''.join(seq))

        random.seed(42)
        random.shuffle(seqs)
        seqs = [s for s in seqs
                if CFG['min_read_len'] <= len(s) <= CFG['max_read_len']][:5000]

        losses = []
        with torch.no_grad():
            for i in range(0, len(seqs), 8):
                batch  = seqs[i:i+8]
                inputs = tokenizer(
                    batch, return_tensors='pt', padding=True,
                    truncation=True, max_length=CFG['max_len'],
                    add_special_tokens=False
                ).to('cuda')
                with autocast('cuda', dtype=torch.float16):
                    out = model(**inputs, labels=inputs['input_ids'])
                logits  = out.logits[...,:-1,:].contiguous().float()
                targets = inputs['input_ids'][...,1:].contiguous()
                mask    = inputs['attention_mask'][...,1:].contiguous().float()
                tl = torch.nn.CrossEntropyLoss(reduction='none')(
                    logits.view(-1, logits.size(-1)), targets.view(-1)
                ).view(targets.size())
                pr = (tl * mask).sum(1) / mask.sum(1).clamp(min=1)
                losses.extend(pr.cpu().float().tolist())
                del out, inputs; torch.cuda.empty_cache()

        mean_ce = float(np.mean(losses))
        std_ce  = float(np.std(losses))
        final_results.append({
            'sample': sample, 'city': city, 'season': season,
            'baseline': BASELINE, 'lora_v6': 4.7890,
            'fullft': mean_ce, 'std': std_ce
        })
        print(f'{sample:<12} {city:<14} {season:<12} {mean_ce:>8.4f} ± {std_ce:.4f}')
    except Exception as e:
        print(f'{sample:<12} {city:<14} {season:<12}   FAILED: {e}')

if final_results:
    mean_fullft  = np.mean([r['fullft'] for r in final_results])
    mean_baseline = np.mean([r['baseline'] for r in final_results])
    print('─' * 65)
    print(f'{"MEAN":<12} {"": <14} {"": <12} {mean_fullft:>8.4f}')
    print()
    print('COMPARISON ACROSS ALL MODEL VERSIONS:')
    print(f'  Baseline METAGENE-1 : {mean_baseline:.4f}')
    print(f'  LoRA v4 (rank 8)    : 4.8046')
    print(f'  LoRA v6 (rank 32)   : 4.7890')
    print(f'  Full fine-tune v7   : {mean_fullft:.4f}  ← THIS EXPERIMENT')
    print(f'  Target (training)   : 1.2400')
    print()
    improvement = mean_baseline - mean_fullft
    print(f'  Improvement vs baseline: +{improvement:.4f} ({improvement/mean_baseline*100:.2f}%)')
    print(f'  Improvement vs LoRA v6: +{4.7890 - mean_fullft:.4f}')
    print(f'  Remaining gap to 1.24 : {mean_fullft - 1.24:.4f}')

In [ ]:
# Cell 9: Save all results
import json
from huggingface_hub import HfApi
api = HfApi()

import pandas as pd
if final_results:
    df = pd.DataFrame(final_results)
    df.to_csv('/content/fullft_val_results.csv', index=False)

with open('/content/fullft_history.json', 'w') as f:
    json.dump({**history, 'final_results': final_results}, f, indent=2)

for fname, hf_path in [
    ('/content/fullft_history.json',   'results/fullft/training_history.json'),
    ('/content/fullft_val_results.csv','results/fullft/val_results.csv'),
    ('/content/fullft_curve.pdf',      'results/fullft/learning_curve.pdf'),
    ('/content/fullft_curve.png',      'results/fullft/learning_curve.png'),
]:
    from pathlib import Path
    if not Path(fname).exists(): continue
    try:
        api.upload_file(
            path_or_fileobj=fname,
            path_in_repo=hf_path,
            repo_id=DATA_REPO, repo_type='dataset', token=HF_TOKEN
        )
        print(f'✓ {hf_path}')
    except Exception as e:
        print(f'✗ {fname}: {e}')

print('\nAll results saved.')
print('Next step: scale to more epochs + all 190M reads')
print('Expected: training loss will approach 1.24 with enough data + epochs')